# Scraper Tempat Makan & Wisata Viral di Indonesia (6 Bulan Terakhir)

Notebook ini melakukan scraping berita dari **Google News RSS** (bahasa Indonesia) untuk mencari artikel tentang tempat makan/wisata yang sedang viral dalam 6 bulan terakhir, lalu mengekstrak:

- Nama tempat
- Lokasi (kelurahan, kecamatan, kota/kabupaten, provinsi) + latitude/longitude (via geocoding OpenStreetMap Nominatim)
- Sejak kapan viral (perkiraan dari tanggal berita/isi artikel)
- Kenapa viral (kalimat yang mengandung alasan viral)
- Sumber berita (judul, link, tanggal publish)

## Catatan penting
- Ekstraksi nama tempat, lokasi, dan alasan viral memakai **heuristik berbasis regex**, bukan NLP/NER penuh, karena berita berbahasa Indonesia sangat bervariasi formatnya. Hasil disarankan **dicek/dibersihkan manual** sebelum dipakai untuk keperluan resmi.
- Geocoding memakai **Nominatim (OpenStreetMap)** yang gratis tapi ada batas 1 request/detik dan mewajibkan `user_agent` yang jelas. Untuk kebutuhan volume besar/produksi, pertimbangkan pakai Google Maps Geocoding API (berbayar) untuk akurasi lebih baik.
- Gunakan data ini untuk riset/analisis internal. Selalu cek Terms of Service situs sumber sebelum melakukan scraping berskala besar.

In [3]:
import json
import re
import time
import urllib.parse
from datetime import datetime, timedelta, timezone

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from googlenewsdecoder import gnewsdecoder
from tqdm.auto import tqdm
import os
import glob

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

c:\Users\PC NEVIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# KEYWORDS = [
#     "tempat makan viral",
#     "kuliner viral",
#     "cafe viral",
#     "restoran viral",
#     "wisata viral",
#     "destinasi wisata viral",
#     "spot wisata viral",
#     "tempat wisata baru viral",
# ]

# MONTHS_BACK = 6           # rentang waktu maksimal (bulan)
# MAX_ITEMS_PER_KEYWORD = 25  # batas jumlah artikel per keyword dari RSS
# REQUEST_DELAY = 1.0        # jeda antar request (detik) - sopan ke server

# CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=30 * MONTHS_BACK)
# print("Ambil artikel sejak:", CUTOFF_DATE.date())

In [5]:
BASE_KEYWORDS = [
    "tempat makan viral",
    "tempat makan enak",
    "kuliner viral",
    "kuliner enak",
    "cafe viral",
    "cafe enak",
    "restoran viral",
    "restoran enak",
    "wisata viral",
    "wisata baru"
]

PROVINCES = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Kepulauan Riau",
    "Jambi", "Sumatera Selatan", "Kepulauan Bangka Belitung", "Bengkulu", "Lampung",
    "DKI Jakarta", "Jawa Barat", "Banten", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur",
    "Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat",
    "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara",
    "Sulawesi Utara", "Gorontalo", "Sulawesi Tengah", "Sulawesi Barat", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Maluku", "Maluku Utara", "Papua", "Papua Barat",
    "Papua Selatan", "Papua Tengah", "Papua Pegunungan", "Papua Barat Daya",
]

BIG_CITIES = [
    "Jakarta Selatan", "Jakarta Pusat", "Jakarta Barat", "Jakarta Timur", "Jakarta Utara",
    "Surabaya", "Bandung", "Medan", "Yogyakarta", "Sleman", "Semarang", "Makassar",
    "Palembang", "Depok", "Bekasi", "Tangerang", "Bogor", "Malang", "Denpasar",
    "Bandar Lampung", "Padang", "Pekanbaru", "Banjarmasin", "Balikpapan", "Samarinda",
    "Pontianak", "Manado", "Jayapura", "Solo", "Cirebon", "Tasikmalaya", "Batam",
]

REGIONAL_CATEGORIES = [
    "tempat makan viral",
    "tempat makan enak",
    "cafe viral",
    "cafe enak",
    "restoran viral",
    "restoran enak",
    "wisata viral",
    "wisata baru",
    # "tempat belanja viral"
]

LOCATION_KEYWORDS = [
    f"{category} {region}"
    for region in PROVINCES + BIG_CITIES
    for category in REGIONAL_CATEGORIES
]
KEYWORDS = list(dict.fromkeys(BASE_KEYWORDS + LOCATION_KEYWORDS))

MONTHS_BACK = 6
MAX_ITEMS_PER_KEYWORD = 100
REQUEST_DELAY = 1.0

CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=30 * MONTHS_BACK)
print("Total keyword:", len(KEYWORDS))
print("Ambil artikel sejak:", CUTOFF_DATE.date())

Total keyword: 570
Ambil artikel sejak: 2026-04-05


In [19]:
EDITIONS = [
    ("id", "ID", "ID:id")
    # ("en-ID", "ID", "ID:en"),
]


def fetch_google_news(keyword: str, max_items: int = 25, hl: str = "id", gl: str = "ID", ceid: str = "ID:id"):
    """Ambil daftar berita dari Google News RSS untuk sebuah keyword & edisi bahasa/negara tertentu."""
    query = urllib.parse.quote(keyword)
    url = f"https://news.google.com/rss/search?q={query}&hl={hl}&gl={gl}&ceid={ceid}"
    feed = feedparser.parse(url)

    items = []
    for entry in feed.entries[:max_items]:
        try:
            pub_date = dateparser.parse(entry.published)
            if pub_date.tzinfo is None:
                pub_date = pub_date.replace(tzinfo=timezone.utc)
        except Exception:
            pub_date = None

        if pub_date is not None and pub_date < CUTOFF_DATE:
            continue  # lewati berita yang lebih lama dari batas waktu

        source = entry.get("source", {}).get("title") if hasattr(entry, "get") else None
        items.append(
            {
                "keyword": keyword,
                "title": entry.title,
                "link": entry.link,
                "published": pub_date,
                "source": source or getattr(getattr(entry, "source", None), "title", None),
            }
        )
    return items


all_news = []
os.makedirs("news", exist_ok=True)
for kw in tqdm(KEYWORDS, desc="Fetch RSS per keyword"):
    for hl, gl, ceid in EDITIONS:
        all_news.extend(fetch_google_news(kw, MAX_ITEMS_PER_KEYWORD, hl=hl, gl=gl, ceid=ceid))
        time.sleep(REQUEST_DELAY)

df_news = pd.DataFrame(all_news).drop_duplicates(subset="link").reset_index(drop=True)
# Simpan semua berita yang telah diambil ke dalam folder "news" tiap keyword
df_news.to_csv("news/all_news.csv", index=False)
for kw in KEYWORDS:
    df_news_kw = df_news[df_news["keyword"] == kw]
    df_news_kw.to_csv(f"news/news_{kw}.csv", index=False)
print(f"Total artikel ditemukan: {len(df_news)}")
df_news

Fetch RSS per keyword:   0%|          | 0/570 [00:00<?, ?it/s]

Fetch RSS per keyword: 100%|ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â | 570/570 [19:28<00:00,  2.05s/it]


Total artikel ditemukan: 5261


,keyword,title,link,published,source
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim
3,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co
4,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
...,...,...,...,...,...
5256,wisata baru Batam,HARRIS Barelang Batam Hadirkan Promo Liburan K...,https://news.google.com/rss/articles/CBMiigFBV...,2026-07-07 07:00:00+00:00,Liputan Bhagasasi
5257,wisata baru Batam,"Pemko Batam Bangun Taman Budaya Rp50 Miliar, T...",https://news.google.com/rss/articles/CBMiyAFBV...,2026-07-15 07:00:00+00:00,Jawa Pos
5258,wisata baru Batam,Disbudpar: 113.784 wisman kunjungi Batam pada ...,https://news.google.com/rss/articles/CBMingFBV...,2026-06-03 07:00:00+00:00,ANTARA News Kepri
5259,wisata baru Batam,"Ascott Resmikan Fox Hotel Nagoya Batam, Hotel ...",https://news.google.com/rss/articles/CBMioAFBV...,2026-05-20 07:00:00+00:00,Beritakota


In [20]:
df_news.keyword.value_counts()

keyword
wisata baru                             99
kuliner viral                           94
wisata viral                            76
tempat makan viral                      73
wisata viral Bali                       68
                                        ..
restoran enak Kalimantan Utara           1
restoran viral Tasikmalaya               1
cafe viral Kepulauan Bangka Belitung     1
cafe enak Kepulauan Bangka Belitung      1
tempat makan viral Manado                1
Name: count, Length: 491, dtype: int64

In [6]:
df_news = pd.read_csv("news/all_news.csv")
df_news 

,keyword,title,link,published,source
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim
3,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co
4,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
...,...,...,...,...,...
5256,wisata baru Batam,HARRIS Barelang Batam Hadirkan Promo Liburan K...,https://news.google.com/rss/articles/CBMiigFBV...,2026-07-07 07:00:00+00:00,Liputan Bhagasasi
5257,wisata baru Batam,"Pemko Batam Bangun Taman Budaya Rp50 Miliar, T...",https://news.google.com/rss/articles/CBMiyAFBV...,2026-07-15 07:00:00+00:00,Jawa Pos
5258,wisata baru Batam,Disbudpar: 113.784 wisman kunjungi Batam pada ...,https://news.google.com/rss/articles/CBMingFBV...,2026-06-03 07:00:00+00:00,ANTARA News Kepri
5259,wisata baru Batam,"Ascott Resmikan Fox Hotel Nagoya Batam, Hotel ...",https://news.google.com/rss/articles/CBMioAFBV...,2026-05-20 07:00:00+00:00,Beritakota


In [7]:
def resolve_and_fetch(url: str, timeout: tuple[int, int] = (5, 15)):
    """Decode URL Google News lalu ambil HTML halaman artikel sumber."""
    source_url = url
    if "news.google.com" in url:
        try:
            decoded = gnewsdecoder(url, interval=1, timeout=10)
            if decoded.get("success") and decoded.get("decoded_url"):
                source_url = decoded["decoded_url"]
        except Exception:
            pass

    try:
        resp = requests.get(source_url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        return resp.url, resp.text
    except requests.RequestException:
        return None, None


def extract_article_text(html: str) -> str:
    """Ambil isi artikel dari JSON-LD, meta description, paragraf, atau body HTML."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")

    # Banyak situs berita menyimpan isi artikel di JSON-LD, bukan tag p.
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            payload = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        candidates = payload if isinstance(payload, list) else [payload]
        for item in candidates:
            if isinstance(item, dict) and isinstance(item.get("articleBody"), str):
                article_body = re.sub(r"\s+", " ", item["articleBody"]).strip()
                if len(article_body) >= 100:
                    return article_body

    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    meta_text = meta.get("content", "").strip() if meta else ""

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [p for p in paragraphs if len(p) >= 20]
    paragraph_text = " ".join(paragraphs)
    if len(paragraph_text) >= 100:
        return paragraph_text
    if len(meta_text) >= 40:
        return meta_text

    body_text = soup.get_text(" ", strip=True) if soup.body else ""
    return re.sub(r"\s+", " ", body_text).strip()


# def fetch_article_batch(
#     df_news: pd.DataFrame,
#     mulai: int = 0,
#     max_articles: int = 2410,
#     request_delay: float = REQUEST_DELAY,
# ) -> tuple[list[str | None], list[str]]:
#     """Fetch dan ekstrak isi artikel sebanyak maksimal `max_articles` baris."""
#     article_texts = []
#     final_urls = []

#     for article_number, link in enumerate(df_news["link"].head(max_articles), start=1):
#         if article_number < mulai: continue
#         print("Processing article number:", article_number)
#         print("Fetching link:", link)
#         final_url, html = resolve_and_fetch(link)
#         final_urls.append(final_url)
#         article_texts.append(extract_article_text(html))
#         print("panjang artikel:", len(article_texts[-1]))
#         time.sleep(request_delay)

#     return final_urls, article_texts


# final_urls, article_texts = fetch_article_batch(df_news, mulai=0, max_articles=2410)

In [8]:
import concurrent.futures

# Kenapa loop suka "diam" tanpa error: gnewsdecoder/requests kadang stall di level
# socket (koneksi macet, bukan timeout HTTP biasa) sehingga thread utama menggantung
# selamanya. fetch_article_safe menjalankan fetch di thread terpisah dan memaksa
# menyerah setelah HARD_TIMEOUT detik, supaya satu link bermasalah tidak mematikan
# seluruh loop.
HARD_TIMEOUT = 25.0
_fetch_executor = concurrent.futures.ThreadPoolExecutor(max_workers=4, thread_name_prefix="fetch")


def fetch_article_safe(link: str, hard_timeout: float = HARD_TIMEOUT):
    """resolve_and_fetch + extract_article_text dengan hard timeout & anti-exception."""
    future = _fetch_executor.submit(resolve_and_fetch, link)
    try:
        final_url, html = future.result(timeout=hard_timeout)
    except Exception:
        return None, ""
    try:
        return final_url, extract_article_text(html)
    except Exception:
        return final_url, ""

In [9]:
import json
import os
from pathlib import Path

# Checkpoint per-baris: setiap link yang selesai difetch langsung ditulis ke file
# ini (flush + fsync). Kalau kernel macet/di-restart, jalankan ulang cell ini ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â
# link yang sudah ada di checkpoint otomatis dilewati (resume), jadi progres tidak
# hilang lagi walau berhenti di tengah jalan.
CHECKPOINT_PATH = Path("fetch_checkpoint.jsonl")

START_FROM = 0
done = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                done[rec["index"]] = rec
            except json.JSONDecodeError:
                continue
    print(f"Resume: {len(done)} link sudah pernah diproses, akan dilewati.")

with CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt_file:
    for idx, link in enumerate(tqdm(df_news[START_FROM:]["link"], desc="Fetch isi artikel")):
        if idx + START_FROM in done:
            continue
        try:
            final_url, text = fetch_article_safe(link)
        except Exception as e:
            print(f"Error tak terduga pada index {idx}: {e}")
            final_url, text = None, ""

        rec = {"index": idx + START_FROM, "link": link, "final_url": final_url, "article_text": text}
        ckpt_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
        ckpt_file.flush()
        os.fsync(ckpt_file.fileno())
        done[idx] = rec
        time.sleep(REQUEST_DELAY)

print(f"Total link diproses (checkpoint): {len(done)} dari {len(df_news)}")

Resume: 5261 link sudah pernah diproses, akan dilewati.


Fetch isi artikel: 100%|ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â | 5261/5261 [00:00<00:00, 3391153.12it/s]

Total link diproses (checkpoint): 5261 dari 5261


In [10]:
# Gabungkan seluruh checkpoint (boleh dijalankan kapan saja, termasuk saat proses
# fetch di atas belum 100% selesai, untuk lihat progres sementara) jadi satu CSV.
indices = sorted(i for i in done if i < len(df_news))
df_result = df_news.iloc[indices].copy()
df_result["final_url"] = [done[i]["final_url"] for i in indices]
df_result["article_text"] = pd.Series(
    [done[i]["article_text"] for i in indices], index=df_result.index, dtype="string"
)
df_result["article_text_length"] = df_result["article_text"].str.len().fillna(0).astype(int)

print(f"Artikel dengan isi berhasil diambil: {(df_result['article_text_length'] >= 100).sum()} dari {len(df_result)}")
print(f"Fetch gagal/kosong: {df_result['article_text'].eq('').sum()}")

df_result_final = df_result[df_result["article_text_length"] >= 100].reset_index(drop=True)
df_result_final.to_csv("df_news_extended_full_coba.csv", index=False)
print("Saved df_news_extended_full_coba.csv")

Artikel dengan isi berhasil diambil: 5090 dari 5261
Fetch gagal/kosong: 97
Saved df_news_extended_full_coba.csv


In [11]:
import glob
import json
import os
import re
import time
from datetime import datetime, timedelta

import pandas as pd
from dateutil import parser as dateparser
from rapidfuzz import fuzz
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

In [5]:
NEWS_CSV_FILE = "df_news_extended_full_coba.csv"
MIN_ARTICLE_LEN = 150

df_news_all = pd.read_csv(NEWS_CSV_FILE)
print("File dipakai:", NEWS_CSV_FILE)

# beberapa batch mungkin punya nama kolom isi artikel yang sedikit berbeda; standardisasi
text_col_candidates = [c for c in ["article_text", "article_text_extended", "text"] if c in df_news_all.columns]
if "article_text" not in df_news_all.columns and text_col_candidates:
    df_news_all["article_text"] = df_news_all[text_col_candidates[0]]

dedup_key = "final_url" if "final_url" in df_news_all.columns else "link"
df_news_all = df_news_all.dropna(subset=[dedup_key])
df_news_all = df_news_all.drop_duplicates(subset=[dedup_key]).reset_index(drop=True)
df_news_all = df_news_all[df_news_all["article_text"].fillna("").str.len() >= MIN_ARTICLE_LEN].reset_index(drop=True)

print("Total artikel unik & layak diproses:", len(df_news_all))
df_news_all.head(3)

File dipakai: df_news_extended_full_coba.csv
Total artikel unik & layak diproses: 4834


,keyword,title,link,published,source,final_url,article_text,article_text_length
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com,https://food.detik.com/tempat-makan/d-8687381/...,Semangkuk ramen berkuah gurih hangat dengan an...,4249
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com,https://jatimnet.com/media-sosial-mengubah-car...,"Jumat, 25 September 2026 08:00 UTC Ilustrasi: ...",7110
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim,https://jatim.idntimes.com/food/dining-guide/1...,Kuliner malam di Sidoarjo menjadi daya tarik t...,7524


## 5. Buat Prompt Batch untuk Ekstraksi Manual via Chat Claude.ai

Sel di bawah membuat file prompt per-batch (`manual_llm_batches_coba/batch_XXX_prompt.txt`) yang bisa langsung ditempel ke chat Claude.ai untuk diekstrak jadi daftar tempat dengan skema kolom final:

`no, nama_tempat, kategori, jenis_tempat, lokasi, keterangan, status_viral, viral_sejak, dasar_viral, kelurahan_desa, kecamatan, kota_kabupaten, provinsi, negara, latitude, longitude, presisi_koordinat, catatan_lokasi, jumlah_artikel, keyword_pencarian, judul_artikel_sumber, link_sumber`

Langkahnya:
1. Jalankan sel "Generate Prompt Batch" ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ini membuat file `batch_001_prompt.txt`, `batch_002_prompt.txt`, dst di folder `manual_llm_batches_coba/`.
2. Buka chat baru di claude.ai, tempel **seluruh isi** satu file `batch_XXX_prompt.txt`, kirim.
3. Simpan balasan JSON Claude (hanya JSON, tanpa teks tambahan) sebagai `manual_llm_batches_coba/batch_XXX_response.json`.
4. Ulangi untuk semua batch, lalu jalankan sel "Gabungkan Hasil ke CSV Final" di bawah untuk membangun tabel final sesuai skema kolom di atas.

Kolom `no`, `jumlah_artikel`, `keyword_pencarian`, `judul_artikel_sumber`, `link_sumber` **tidak** diminta dari Claude ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â `jumlah_artikel` dihitung otomatis saat tempat yang sama disebut di beberapa artikel (fuzzy dedup berdasarkan nama), sisanya diisi dari metadata artikel saat hasil digabungkan.

In [13]:
MANUAL_BATCH_DIR = "manual_llm_batches_coba"
MANUAL_BATCH_SIZE = 8       # jumlah artikel per file prompt - kecilkan jika artikel panjang
MANUAL_MAX_CHARS = 3000     # potong isi artikel supaya prompt tidak terlalu panjang untuk chat

In [ ]:
os.makedirs(MANUAL_BATCH_DIR, exist_ok=True)

# simpan pemetaan article_id -> metadata artikel, dipakai lagi saat mengimpor hasil balasan
id_map = {}
for idx, row in df_news_all.iterrows():
    id_map[str(idx)] = {
        "final_url": row[dedup_key],
        "title": row.get("title", ""),
        "link": row.get("link", row[dedup_key]),
        "keyword": row.get("keyword", ""),
    }
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

PROMPT_HEADER = """Kamu akan menerima beberapa artikel berita. Untuk SETIAP artikel, ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan.

Balas HANYA dengan JSON valid (tanpa penjelasan, tanpa markdown code fence) dengan format:
{
  "results": [
    {
      "article_id": <id artikel, angka>,
      "places": [
        {
          "nama_tempat": "...",
          "kategori": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya",
          "jenis_tempat": "subtipe lebih spesifik, mis. 'rooftop cafe', 'kuliner kaki lima', 'air terjun', 'taman hiburan', dst",
          "lokasi": "lokasi mentah apa adanya sesuai artikel (alamat/nama jalan/area jika disebut)",
          "keterangan": "ringkasan singkat ciri khas/daya tarik tempat ini (mis. menu andalan, konsep, fasilitas)",
          "status_viral": "Disebut viral" (artikel eksplisit menyebut tempat ini viral/ramai dibahas/jadi perbincangan warganet) atau "Rekomendasi" (tempat hanya direkomendasikan/masuk daftar, tanpa disebut eksplisit viral),
          "viral_sejak": "tanggal/bulan/periode eksplisit terkait mulai viral jika disebut, atau 'Tidak disebut' jika tidak ada",
          "dasar_viral": "kutipan/alasan singkat dari artikel yang menjadi dasar kenapa tempat ini dianggap viral/direkomendasikan, atau null jika tidak ada dasar eksplisit",
          "kelurahan_desa": "nama kelurahan/desa, atau null jika tidak diketahui",
          "kecamatan": "nama kecamatan, atau null jika tidak diketahui",
          "kota_kabupaten": "nama kota/kabupaten, atau null jika tidak diketahui",
          "provinsi": "nama provinsi, atau null jika tidak diketahui",
          "negara": "nama negara, default 'Indonesia' kecuali artikel jelas menyebut negara lain",
          "latitude": angka desimal lintang jika kamu bisa simpulkan/cari dari nama tempat & lokasi, atau null jika tidak yakin,
          "longitude": angka desimal bujur jika kamu bisa simpulkan/cari dari nama tempat & lokasi, atau null jika tidak yakin,
          "presisi_koordinat": "exact" (alamat persis diketahui), "approximate" (hanya area/kecamatan/kota), atau "unknown" (tidak ada dasar koordinat),
          "catatan_lokasi": "catatan tambahan soal lokasi, mis. ketidakpastian, patokan terdekat, atau null jika tidak ada"
        }
      ]
    }
  ]
}

Jika sebuah artikel tidak menyebut tempat spesifik, kembalikan places: [] untuk artikel itu. Sertakan SEMUA article_id yang diberikan di bawah, jangan ada yang terlewat.
"""

article_ids = list(df_news_all.index)
batches = [article_ids[i:i + MANUAL_BATCH_SIZE] for i in range(0, len(article_ids), MANUAL_BATCH_SIZE)]

for b_num, batch_ids in enumerate(batches, start=1):
    parts = [PROMPT_HEADER, "\nArtikel:\n"]
    for aid in batch_ids:
        row = df_news_all.loc[aid]
        text = str(row.get("article_text", ""))[:MANUAL_MAX_CHARS]
        parts.append(f"\n### Artikel {aid}\nJudul: {row.get('title', '')}\nIsi:\n{text}\n")
    prompt_text = "\n".join(parts)
    fname = os.path.join(MANUAL_BATCH_DIR, f"batch_{b_num:03d}_prompt.txt")
    with open(fname, "w", encoding="utf-8") as f:
        f.write(prompt_text)

print(f"Dibuat {len(batches)} file prompt di folder '{MANUAL_BATCH_DIR}/'")

Dibuat 605 file prompt di folder 'manual_llm_batches_coba/'


**Setelah semua file `batch_XXX_response.json` tersimpan** di folder `manual_llm_batches_coba/` (isi tiap file harus JSON valid sesuai format di atas), jalankan sel berikut untuk menggabungkan semuanya jadi tabel final dengan kolom sesuai skema yang diminta.

In [14]:
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), encoding="utf-8") as f:
    id_map = json.load(f)

RESPONSE_BATCH_DIR = "responses_json_coba"
response_files = sorted(glob.glob(os.path.join(RESPONSE_BATCH_DIR, "batch_*_response.json")))
print("File respons ditemukan:", len(response_files))

rows = []
for rf in response_files:
    with open(rf, encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            print("Gagal parse", rf, "-> pastikan isinya JSON valid tanpa teks tambahan:", e)
            continue
    for result in data.get("results", []):
        aid = str(result.get("article_id"))
        meta = id_map.get(aid)
        if not meta:
            print("article_id tidak dikenal:", aid, "di", rf)
            continue
        for p in result.get("places", []):
            rows.append({
                "nama_tempat": p.get("nama_tempat"),
                "kategori": p.get("kategori"),
                "jenis_tempat": p.get("jenis_tempat"),
                "lokasi": p.get("lokasi"),
                "keterangan": p.get("keterangan"),
                "status_viral": p.get("status_viral"),
                "viral_sejak": p.get("viral_sejak"),
                "dasar_viral": p.get("dasar_viral"),
                "kelurahan_desa": p.get("kelurahan_desa"),
                "kecamatan": p.get("kecamatan"),
                "kota_kabupaten": p.get("kota_kabupaten"),
                "provinsi": p.get("provinsi"),
                "negara": p.get("negara"),
                "latitude": p.get("latitude"),
                "longitude": p.get("longitude"),
                "presisi_koordinat": p.get("presisi_koordinat"),
                "catatan_lokasi": p.get("catatan_lokasi"),
                "_article_key": aid,
                "keyword_pencarian": meta.get("keyword"),
                "judul_artikel_sumber": meta.get("title"),
                "link_sumber": meta.get("link") or meta.get("final_url"),
            })

df_places_raw = pd.DataFrame(rows).dropna(subset=["nama_tempat"]).reset_index(drop=True)
print("Total baris tempat mentah (sebelum dedup antar-artikel):", len(df_places_raw))
df_places_raw.head(5)

File respons ditemukan: 170
Total baris tempat mentah (sebelum dedup antar-artikel): 2467


,nama_tempat,kategori,jenis_tempat,lokasi,keterangan,status_viral,viral_sejak,dasar_viral,kelurahan_desa,kecamatan,...,provinsi,negara,latitude,longitude,presisi_koordinat,catatan_lokasi,_article_key,keyword_pencarian,judul_artikel_sumber,link_sumber
0,Sai Ramen,Restoran,restoran ramen Jepang halal,"Sampoerna Strategic Building, gedung Synergy, Living World Kota Wisata","Ramen halal harga bersahabat Rp50-75 ribu; Black Garlic Ramen, Akakara Spicy Ramen, kaldu ayam pekat",Rekomendasi,Tidak disebut,Ramen bersertifikat halal dengan harga bersahabat,None,Setiabudi,...,DKI Jakarta,Indonesia,-6.2196,106.8186,approximate,Punya beberapa cabang; koordinat mengacu ke cabang Sampoerna Strategic Square,0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
1,Menya Beast,Restoran,restoran ramen Jepang,"Lantai Dasar Trinity Tower, Setiabudi, Kuningan","Resep chef Jepang, kaldu dimasak 12 jam; Jun Tonkotsu Mayu Ramen, Taiwan Tantan Abura Soba; Rp100-130 ribu",Rekomendasi,Tidak disebut,"Ramen dengan resep chef asal Jepang, kaldu dimasak 12 jam",Karet Kuningan,Setiabudi,...,DKI Jakarta,Indonesia,-6.2187,106.8303,approximate,"Di dalam gedung Trinity Tower, Kuningan",0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
2,Raku Ramen,Restoran,kedai ramen Jepang (basement),"Fairground SCBD, Senayan, Kebayoran Baru, Jakarta Selatan","Gerai 'nyempil' di bawah tanah, kuah shoyu/shio/paitan, mulai Rp50 ribuan",Rekomendasi,Tidak disebut,Lokasi nyempil di bawah tanah tetapi tidak pernah sepi pengunjung,Senayan,Kebayoran Baru,...,DKI Jakarta,Indonesia,-6.2257,106.8089,approximate,Area Fairground SCBD,0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
3,Echigoya,Restoran,restoran ramen Jepang autentik,None,"Ramen autentik langganan warga Jepang, kuah kental milky, topping ayam atau babi",Rekomendasi,Tidak disebut,Sering menjadi tujuan banyak warga lokal Jepang,None,None,...,DKI Jakarta,Indonesia,NaN,NaN,unknown,Lokasi terpotong di artikel; hanya disebut di Jakarta,0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
4,Warung Amanda,Restoran,warung makan (sego sambel lento),"Jl. Raya Taman Pinang Indah No.6, Banjarbendo, Kec. Sidoarjo","Sego sambel lento khas Sidoarjo, buka 24 jam, mulai Rp11 ribu, hanya tunai",Rekomendasi,Tidak disebut,"Menjual sego sambel lento, menu khas Sidoarjo",Banjarbendo,Sidoarjo,...,Jawa Timur,Indonesia,-7.4560,112.7010,approximate,Kawasan Taman Pinang Indah,2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Takut Lapar Saat Begadang - IDN Times Jatim",https://news.google.com/rss/articles/CBMizwFBVV95cUxQbG9QZ2VwZ3NYR3lVb1N2Z25GTkkxWU93Tko3SGtkakhXN2JVVmFhXzRJc1NsSWp...


Tempat yang sama bisa disebut di beberapa artikel berbeda (dengan penulisan nama sedikit berbeda). Sel berikut melakukan **fuzzy dedup** nama tempat (`rapidfuzz.fuzz.token_sort_ratio`) supaya tiap tempat jadi satu baris, lalu menghitung `jumlah_artikel` dan menyimpan ke CSV final dengan urutan kolom sesuai skema.

In [15]:
FUZZY_THRESHOLD = 88  # 0-100, makin tinggi makin ketat kemiripan nama

FINAL_COLUMNS = [
    "no", "nama_tempat", "kategori", "jenis_tempat", "lokasi", "keterangan",
    "status_viral", "viral_sejak", "dasar_viral", "kelurahan_desa", "kecamatan",
    "kota_kabupaten", "provinsi", "negara", "latitude", "longitude",
    "presisi_koordinat", "catatan_lokasi", "jumlah_artikel",
    "keyword_pencarian", "judul_artikel_sumber", "link_sumber",
]


def normalize_place_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def first_non_null(series: pd.Series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) else None


df_places_raw["_nama_norm"] = df_places_raw["nama_tempat"].map(normalize_place_name)

# cluster nama unik yang mirip (greedy): tiap nama unik dibandingkan ke representative cluster yang sudah ada
unique_names = df_places_raw["_nama_norm"].unique().tolist()
name_to_group = {}
cluster_reps = []  # representative nama tiap cluster, indexnya = id cluster

for name in unique_names:
    if not name:
        continue
    best_idx, best_score = None, 0
    for idx, rep in enumerate(cluster_reps):
        score = fuzz.token_sort_ratio(name, rep)
        if score > best_score:
            best_idx, best_score = idx, score
    if best_score >= FUZZY_THRESHOLD:
        name_to_group[name] = best_idx
    else:
        cluster_reps.append(name)
        name_to_group[name] = len(cluster_reps) - 1

df_places_raw["canonical_group"] = df_places_raw["_nama_norm"].map(name_to_group)
print("Total nama mentah unik:", len(unique_names))
print("Total tempat unik setelah fuzzy dedup:", df_places_raw["canonical_group"].nunique())

agg_cols = [c for c in FINAL_COLUMNS if c not in ("no", "jumlah_artikel")]
grouped_rows = []
for _, g in df_places_raw.groupby("canonical_group", sort=False):
    row = {col: first_non_null(g[col]) for col in agg_cols if col in g.columns}
    row["jumlah_artikel"] = g["_article_key"].nunique()
    grouped_rows.append(row)

df_places_final = pd.DataFrame(grouped_rows, columns=[c for c in FINAL_COLUMNS if c != "no"])
df_places_final.insert(0, "no", range(1, len(df_places_final) + 1))

# OUTPUT_CSV = "hasil_ekstraksi_tempat_viral.csv"
# df_places_final.to_csv(OUTPUT_CSV, index=False)
print(f"Total baris tempat (setelah dedup): {len(df_places_final)}")
# print(f"Total baris tempat (setelah dedup): {len(df_places_final)}. Disimpan ke {OUTPUT_CSV}")
df_places_final.head(5)

Total nama mentah unik: 2082
Total tempat unik setelah fuzzy dedup: 2051
Total baris tempat (setelah dedup): 2051


,no,nama_tempat,kategori,jenis_tempat,lokasi,keterangan,status_viral,viral_sejak,dasar_viral,kelurahan_desa,...,provinsi,negara,latitude,longitude,presisi_koordinat,catatan_lokasi,jumlah_artikel,keyword_pencarian,judul_artikel_sumber,link_sumber
0,1,Sai Ramen,Restoran,restoran ramen Jepang halal,"Sampoerna Strategic Building, gedung Synergy, Living World Kota Wisata","Ramen halal harga bersahabat Rp50-75 ribu; Black Garlic Ramen, Akakara Spicy Ramen, kaldu ayam pekat",Rekomendasi,Tidak disebut,Ramen bersertifikat halal dengan harga bersahabat,Karet Semanggi,...,DKI Jakarta,Indonesia,-6.2196,106.8186,approximate,Punya beberapa cabang; koordinat mengacu ke cabang Sampoerna Strategic Square,2,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
1,2,Menya Beast,Restoran,restoran ramen Jepang,"Lantai Dasar Trinity Tower, Setiabudi, Kuningan","Resep chef Jepang, kaldu dimasak 12 jam; Jun Tonkotsu Mayu Ramen, Taiwan Tantan Abura Soba; Rp100-130 ribu",Rekomendasi,Tidak disebut,"Ramen dengan resep chef asal Jepang, kaldu dimasak 12 jam",Karet Kuningan,...,DKI Jakarta,Indonesia,-6.2187,106.8303,approximate,"Di dalam gedung Trinity Tower, Kuningan",1,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
2,3,Raku Ramen,Restoran,kedai ramen Jepang (basement),"Fairground SCBD, Senayan, Kebayoran Baru, Jakarta Selatan","Gerai 'nyempil' di bawah tanah, kuah shoyu/shio/paitan, mulai Rp50 ribuan",Rekomendasi,Tidak disebut,Lokasi nyempil di bawah tanah tetapi tidak pernah sepi pengunjung,Senayan,...,DKI Jakarta,Indonesia,-6.2257,106.8089,approximate,Area Fairground SCBD,2,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
3,4,Echigoya,Restoran,restoran ramen Jepang autentik,None,"Ramen autentik langganan warga Jepang, kuah kental milky, topping ayam atau babi",Rekomendasi,Tidak disebut,Sering menjadi tujuan banyak warga lokal Jepang,None,...,DKI Jakarta,Indonesia,NaN,NaN,unknown,Lokasi terpotong di artikel; hanya disebut di Jakarta,1,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
4,5,Warung Amanda,Restoran,warung makan (sego sambel lento),"Jl. Raya Taman Pinang Indah No.6, Banjarbendo, Kec. Sidoarjo","Sego sambel lento khas Sidoarjo, buka 24 jam, mulai Rp11 ribu, hanya tunai",Rekomendasi,Tidak disebut,"Menjual sego sambel lento, menu khas Sidoarjo",Banjarbendo,...,Jawa Timur,Indonesia,-7.4560,112.7010,approximate,Kawasan Taman Pinang Indah,1,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Takut Lapar Saat Begadang - IDN Times Jatim",https://news.google.com/rss/articles/CBMizwFBVV95cUxQbG9QZ2VwZ3NYR3lVb1N2Z25GTkkxWU93Tko3SGtkakhXN2JVVmFhXzRJc1NsSWp...


In [16]:
df_places_final

,no,nama_tempat,kategori,jenis_tempat,lokasi,keterangan,status_viral,viral_sejak,dasar_viral,kelurahan_desa,...,provinsi,negara,latitude,longitude,presisi_koordinat,catatan_lokasi,jumlah_artikel,keyword_pencarian,judul_artikel_sumber,link_sumber
0,1,Sai Ramen,Restoran,restoran ramen Jepang halal,"Sampoerna Strategic Building, gedung Synergy, Living World Kota Wisata","Ramen halal harga bersahabat Rp50-75 ribu; Black Garlic Ramen, Akakara Spicy Ramen, kaldu ayam pekat",Rekomendasi,Tidak disebut,Ramen bersertifikat halal dengan harga bersahabat,Karet Semanggi,...,DKI Jakarta,Indonesia,-6.2196,106.8186,approximate,Punya beberapa cabang; koordinat mengacu ke cabang Sampoerna Strategic Square,2,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
1,2,Menya Beast,Restoran,restoran ramen Jepang,"Lantai Dasar Trinity Tower, Setiabudi, Kuningan","Resep chef Jepang, kaldu dimasak 12 jam; Jun Tonkotsu Mayu Ramen, Taiwan Tantan Abura Soba; Rp100-130 ribu",Rekomendasi,Tidak disebut,"Ramen dengan resep chef asal Jepang, kaldu dimasak 12 jam",Karet Kuningan,...,DKI Jakarta,Indonesia,-6.2187,106.8303,approximate,"Di dalam gedung Trinity Tower, Kuningan",1,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
2,3,Raku Ramen,Restoran,kedai ramen Jepang (basement),"Fairground SCBD, Senayan, Kebayoran Baru, Jakarta Selatan","Gerai 'nyempil' di bawah tanah, kuah shoyu/shio/paitan, mulai Rp50 ribuan",Rekomendasi,Tidak disebut,Lokasi nyempil di bawah tanah tetapi tidak pernah sepi pengunjung,Senayan,...,DKI Jakarta,Indonesia,-6.2257,106.8089,approximate,Area Fairground SCBD,2,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
3,4,Echigoya,Restoran,restoran ramen Jepang autentik,None,"Ramen autentik langganan warga Jepang, kuah kental milky, topping ayam atau babi",Rekomendasi,Tidak disebut,Sering menjadi tujuan banyak warga lokal Jepang,None,...,DKI Jakarta,Indonesia,NaN,NaN,unknown,Lokasi terpotong di artikel; hanya disebut di Jakarta,1,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang Jepang hingga Selebgram - food.detik.com,https://news.google.com/rss/articles/CBMitAFBVV95cUxPQUJIYUdRRG8xWkJUdXhNT2o0SnlTVHQ2TXNfdlBodFFmNk5IT0haOEo1SGNLYUZ...
4,5,Warung Amanda,Restoran,warung makan (sego sambel lento),"Jl. Raya Taman Pinang Indah No.6, Banjarbendo, Kec. Sidoarjo","Sego sambel lento khas Sidoarjo, buka 24 jam, mulai Rp11 ribu, hanya tunai",Rekomendasi,Tidak disebut,"Menjual sego sambel lento, menu khas Sidoarjo",Banjarbendo,...,Jawa Timur,Indonesia,-7.4560,112.7010,approximate,Kawasan Taman Pinang Indah,1,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Takut Lapar Saat Begadang - IDN Times Jatim",https://news.google.com/rss/articles/CBMizwFBVV95cUxQbG9QZ2VwZ3NYR3lVb1N2Z25GTkkxWU93Tko3SGtkakhXN2JVVmFhXzRJc1NsSWp...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2046,2047,Klyf Club,Kafe,beach club kolam infinity,"Blue Lagoon Avia Villas, Nusa Ceningan, Klungkung","Di Blue Lagoon Avia Villas, kolam infinity menghadap Blue Lagoon",Rekomendasi,Tidak disebut,Direkomendasikan di panduan wisata Nusa Ceningan,None,...,Bali,Indonesia,-8.6950,115.4560,approximate,Titik di pulau kecil Nusa Ceningan; koordinat perkiraan,1,wisata viral Bali,"Panduan Wisata ke Nusa Ceningan, Hidden Gem Bali! ÃƒÆ’Ã‚Â°Ãƒâ€¦Ã‚Â¸ÃƒÂ¢Ã¢â€šÂ¬Ã‚ÂºÃƒâ€šÃ‚Âµ - TripZilla Indonesia",https://news.google.com/rss/articles/CBMiekFVX3lxTE9zNWlwNjF2enI3UXVaSzIxbVdaSnZ5NXhNcjctZHJZM2RKUmRLOE01ajlreHpER3J...
2047,2048,The 

In [17]:
# # Hitung jumlah berita 6 bulan terakhir per nama_tempat (Google News RSS), plus
# # tanggal berita pertama, terakhir, dan selisih harinya. Hasil ditambah ke df_places_final.
# # Keyword = "nama_tempat" + kota_kabupaten (kalau ada) supaya nama umum tidak tercampur tempat lain.
# # Checkpoint per tempat ke JSONL -> aman di-resume kalau kernel berhenti.
# import concurrent.futures
# import threading
# from pathlib import Path

# PLACE_CKPT = Path("place_news_stats_checkpoint.jsonl")
# PLACE_DELAY = 1.0          # jeda per request (detik), per worker
# PLACE_WORKERS = 3          # jangan terlalu besar supaya tidak kena rate limit Google
# PLACE_MAX_ITEMS = 100      # RSS Google News maksimal ~100 item per query
# PLACE_DAYS_BACK = 180      # ~6 bulan
# USE_CITY_IN_QUERY = True   # False = cari hanya berdasarkan nama_tempat

# place_cutoff = datetime.now(timezone.utc) - timedelta(days=PLACE_DAYS_BACK)


# def fetch_place_news_stats(place: str, city: str | None = None, retries: int = 3) -> dict:
#     """Cari berita dengan keyword "nama_tempat" [kota] dalam 6 bulan terakhir."""
#     q = f'"{place}"'
#     if USE_CITY_IN_QUERY and city:
#         q += f' "{city}"'
#     query = urllib.parse.quote(f"{q} when:{PLACE_DAYS_BACK}d")
#     url = f"https://news.google.com/rss/search?q={query}&hl=id&gl=ID&ceid=ID:id"

#     entries = None
#     for attempt in range(retries):
#         try:
#             resp = requests.get(url, headers=HEADERS, timeout=20)
#             if resp.status_code == 200:
#                 entries = feedparser.parse(resp.content).entries
#                 break
#         except requests.RequestException:
#             pass
#         time.sleep(3 * (attempt + 1))
#     if entries is None:
#         return {"nama_tempat": place, "error": True}

#     dates, seen = [], set()
#     for e in entries[:PLACE_MAX_ITEMS]:
#         key = e.get("link") or e.get("title")
#         if key in seen:
#             continue
#         seen.add(key)
#         try:
#             d = dateparser.parse(e.published)
#             if d.tzinfo is None:
#                 d = d.replace(tzinfo=timezone.utc)
#         except Exception:
#             continue
#         if d >= place_cutoff:
#             dates.append(d)

#     return {
#         "nama_tempat": place,
#         "error": False,
#         "keyword_berita": q,
#         "jumlah_berita_6bln": len(dates),
#         "tanggal_berita_pertama": min(dates).date().isoformat() if dates else None,
#         "tanggal_berita_terakhir": max(dates).date().isoformat() if dates else None,
#     }


# # --- resume dari checkpoint (baris error akan dicoba ulang) ---
# place_done = {}
# if PLACE_CKPT.exists():
#     with PLACE_CKPT.open(encoding="utf-8") as f:
#         for line in f:
#             try:
#                 rec = json.loads(line)
#             except json.JSONDecodeError:
#                 continue
#             if not rec.get("error"):
#                 place_done[rec["nama_tempat"]] = rec

# # satu pasangan (nama_tempat, kota) per nama unik
# df_todo = (
#     df_places_final.dropna(subset=["nama_tempat"])
#     .drop_duplicates(subset="nama_tempat")[["nama_tempat", "kota_kabupaten"]]
# )
# todo = [
#     (p, c if isinstance(c, str) and c.strip() else None)
#     for p, c in df_todo.itertuples(index=False)
#     if p not in place_done
# ]
# print(f"Sudah ada: {len(place_done)} | Perlu diproses: {len(todo)}")

# _lock = threading.Lock()


# def _worker(args):
#     place, city = args
#     rec = fetch_place_news_stats(place, city)
#     time.sleep(PLACE_DELAY)
#     return rec


# with PLACE_CKPT.open("a", encoding="utf-8") as ckpt, \
#         concurrent.futures.ThreadPoolExecutor(max_workers=PLACE_WORKERS) as ex:
#     futures = [ex.submit(_worker, t) for t in todo]
#     for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Berita per tempat"):
#         rec = fut.result()
#         with _lock:
#             ckpt.write(json.dumps(rec, ensure_ascii=False) + "\n")
#             ckpt.flush()
#         if not rec.get("error"):
#             place_done[rec["nama_tempat"]] = rec

# # --- gabungkan ke df_places_final ---
# stat_cols = ["jumlah_berita_6bln", "tanggal_berita_pertama", "tanggal_berita_terakhir"]
# df_stats = pd.DataFrame(place_done.values())[["nama_tempat", "keyword_berita"] + stat_cols]
# df_places_final = df_places_final.drop(
#     columns=[c for c in df_stats.columns if c != "nama_tempat"], errors="ignore"
# ).merge(df_stats, on="nama_tempat", how="left")

# tgl_pertama = pd.to_datetime(df_places_final["tanggal_berita_pertama"])
# tgl_terakhir = pd.to_datetime(df_places_final["tanggal_berita_terakhir"])
# df_places_final["selisih_hari"] = (tgl_terakhir - tgl_pertama).dt.days

# gagal = df_places_final["jumlah_berita_6bln"].isna().sum()
# print(f"Tempat tanpa data (gagal fetch, jalankan ulang cell ini): {gagal}")
# df_places_final[["nama_tempat", "keyword_berita"] + stat_cols + ["selisih_hari"]].head(10)


In [19]:
# Versi Google Search (tab Berita): https://www.google.com/search?q={query}&tbm=nws&hl=id&gl=ID
# Hitung jumlah berita 6 bulan terakhir per nama_tempat + tanggal pertama, terakhir, selisih hari
# -> menimpa kolom hasil dari versi RSS di df_places_final.
#
# Butuh:  pip install selenium   (+ Google Chrome; driver diunduh otomatis oleh Selenium Manager)
# Google minta JavaScript, jadi pakai browser asli (jendela Chrome terbuka, jangan ditutup).
# Kalau muncul CAPTCHA: selesaikan manual di jendela Chrome, script lanjut otomatis.
# Checkpoint per tempat ke JSONL -> aman di-resume kalau berhenti.
import random
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

PLACE_CKPT = Path("place_news_stats_google_checkpoint.jsonl")
PLACE_DAYS_BACK = 180          # ~6 bulan
MAX_PAGES = 3                  # halaman hasil per tempat (~10 berita/halaman) -> jumlah maksimal ~30
DELAY_RANGE = (3.0, 6.0)       # jeda acak antar request (detik) supaya tidak kena blokir
USE_CITY_IN_QUERY = True       # False = cari hanya berdasarkan nama_tempat
CAPTCHA_WAIT_SECONDS = 600     # batas waktu menunggu CAPTCHA diselesaikan manual
LIMIT = 100                      # tes dulu 5 tempat, cek hasil, lalu ubah ke None untuk semua

today = datetime.now()
range_start = today - timedelta(days=PLACE_DAYS_BACK)

MONTHS_ID = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "jun": 6, "jul": 7,
             "agu": 8, "agt": 8, "sep": 9, "okt": 10, "nov": 11, "des": 12}
UNIT_DAYS = {"menit": 0, "jam": 0, "hari": 1, "minggu": 7, "bulan": 30, "tahun": 365}
REL_RE = re.compile(r"^(\d+)\s+(menit|jam|hari|minggu|bulan|tahun)\s+(?:yang\s+)?lalu$", re.I)
ABS_RE = re.compile(
    r"^(\d{1,2})\s+(jan|feb|mar|apr|mei|jun|jul|agu|agt|sep|okt|nov|des)[a-z]*\.?(?:\s+(\d{4}))?$", re.I
)


def parse_news_date(text: str):
    """Parse teks tanggal Google ('3 hari yang lalu', '5 Mei 2026', 'Kemarin') -> datetime, atau None."""
    t = text.strip()
    if t.lower() == "kemarin":
        return today - timedelta(days=1)
    m = REL_RE.match(t)
    if m:
        return today - timedelta(days=int(m.group(1)) * UNIT_DAYS[m.group(2).lower()])
    m = ABS_RE.match(t)
    if m:
        year = int(m.group(3)) if m.group(3) else today.year
        try:
            d = datetime(year, MONTHS_ID[m.group(2).lower()[:3]], int(m.group(1)))
        except ValueError:
            return None
        if not m.group(3) and d > today:
            d = d.replace(year=d.year - 1)
        return d
    return None


def build_google_news_url(query: str, start: int = 0) -> str:
    cd_min = f"{range_start.month}/{range_start.day}/{range_start.year}"
    cd_max = f"{today.month}/{today.day}/{today.year}"
    url = (
        f"https://www.google.com/search?q={urllib.parse.quote(query)}&tbm=nws&hl=id&gl=ID"
        f"&tbs=cdr:1,cd_min:{cd_min},cd_max:{cd_max}"
    )
    return url + (f"&start={start}" if start else "")


def parse_results(html: str) -> dict:
    """Ambil {link: tanggal} tiap kartu berita. Link hasil di Google berbentuk '/goto?url=...'
    (atau http langsung); teks tanggal ('22 Apr 2026', '43 menit yang lalu') ada di dalam <a> yang sama.
    Tidak bergantung pada nama class CSS Google."""
    soup = BeautifulSoup(html, "lxml")
    found = {}
    for a in soup.find_all("a", href=True):
        href = a["href"]
        is_result = href.startswith("/goto?url=") or (href.startswith("http") and "google." not in href)
        if not is_result:
            continue
        for el in a.find_all(["span", "div"]):
            txt = el.get_text(" ", strip=True)
            if len(txt) <= 25:
                date = parse_news_date(txt)
                if date:
                    found.setdefault(href, date)
                    break
    return found


def make_driver():
    opts = Options()
    opts.add_argument("--lang=id-ID")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument(f"--user-data-dir={Path('chrome_profile_news').resolve()}")  # simpan cookie antar sesi
    # chromedriver lama di PATH (mis. C:\\Selenium, v112) bikin SessionNotCreatedException;
    # buang dari PATH supaya Selenium Manager mengunduh driver yang cocok dengan versi Chrome.
    os.environ["PATH"] = os.pathsep.join(
        p for p in os.environ.get("PATH", "").split(os.pathsep)
        if not os.path.exists(os.path.join(p, "chromedriver.exe"))
    )
    return webdriver.Chrome(options=opts)


def wait_if_blocked(driver):
    """Kalau Google menampilkan CAPTCHA/'unusual traffic', tunggu sampai diselesaikan manual."""
    waited = 0
    while "/sorry/" in driver.current_url or "unusual traffic" in driver.page_source.lower():
        if waited == 0:
            print("CAPTCHA terdeteksi -> selesaikan manual di jendela Chrome...")
        if waited >= CAPTCHA_WAIT_SECONDS:
            raise RuntimeError("CAPTCHA tidak diselesaikan; jalankan ulang cell ini nanti (progres tersimpan).")
        time.sleep(5)
        waited += 5


def clean_city(city: str | None) -> str | None:
    if not isinstance(city, str) or not city.strip():
        return None
    return re.sub(r"^(kota|kabupaten|kab\.)\s+", "", city.strip(), flags=re.I)


def fetch_place_news_stats(driver, place: str, city: str | None = None) -> dict:
    q = f'{place}'
    if USE_CITY_IN_QUERY and city:
        q += f' {city}'

    all_found = {}
    for page in range(MAX_PAGES):
        driver.get(build_google_news_url(q, start=page * 10))
        time.sleep(1.5)
        wait_if_blocked(driver)
        found = parse_results(driver.page_source)
        new = {k: v for k, v in found.items() if k not in all_found}
        all_found.update(new)
        if not new or len(found) < 10:  # halaman terakhir
            break
        time.sleep(random.uniform(*DELAY_RANGE))

    dates = [d for d in all_found.values() if d >= range_start]
    return {
        "nama_tempat": place,
        "error": False,
        "keyword_berita": q,
        "jumlah_berita_6bln": len(dates),
        "tanggal_berita_pertama": min(dates).date().isoformat() if dates else None,
        "tanggal_berita_terakhir": max(dates).date().isoformat() if dates else None,
    }


# --- resume dari checkpoint ---
place_done = {}
if PLACE_CKPT.exists():
    with PLACE_CKPT.open(encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not rec.get("error"):
                place_done[rec["nama_tempat"]] = rec

df_todo = (
    df_places_final.dropna(subset=["nama_tempat"])
    .drop_duplicates(subset="nama_tempat")[["nama_tempat", "kota_kabupaten"]]
)
todo = [(p, clean_city(c)) for p, c in df_todo.itertuples(index=False) if p not in place_done]
print(f"Sudah ada: {len(place_done)} | Perlu diproses: {len(todo)}")
if LIMIT:
    todo = todo[:LIMIT]

if todo:
    driver = make_driver()
    try:
        with PLACE_CKPT.open("a", encoding="utf-8") as ckpt:
            for place, city in tqdm(todo, desc="Berita per tempat (Google)"):
                try:
                    rec = fetch_place_news_stats(driver, place, city)
                except RuntimeError:
                    raise
                except Exception as e:
                    print(f"Gagal '{place}': {e}")
                    rec = {"nama_tempat": place, "error": True}
                ckpt.write(json.dumps(rec, ensure_ascii=False) + "\n")
                ckpt.flush()
                if not rec.get("error"):
                    place_done[place] = rec
                time.sleep(random.uniform(*DELAY_RANGE))
    finally:
        driver.quit()

# --- gabungkan ke df_places_final (menimpa hasil RSS) ---
stat_cols = ["jumlah_berita_6bln", "tanggal_berita_pertama", "tanggal_berita_terakhir"]
df_stats = pd.DataFrame(place_done.values())[["nama_tempat", "keyword_berita"] + stat_cols]
df_places_final = df_places_final.drop(
    columns=[c for c in df_stats.columns if c != "nama_tempat"] + ["selisih_hari"], errors="ignore"
).merge(df_stats, on="nama_tempat", how="left")

tgl_pertama = pd.to_datetime(df_places_final["tanggal_berita_pertama"])
tgl_terakhir = pd.to_datetime(df_places_final["tanggal_berita_terakhir"])
df_places_final["selisih_hari"] = (tgl_terakhir - tgl_pertama).dt.days

print(f"Tempat belum punya data: {df_places_final['jumlah_berita_6bln'].isna().sum()}")
# save csv
# df_places_final.to_csv("df_places_final_with_news_stats.csv", index=False)
df_places_final[["nama_tempat", "keyword_berita"] + stat_cols + ["selisih_hari"]].dropna(
    subset=["jumlah_berita_6bln"]
)


Sudah ada: 39 | Perlu diproses: 2012


Berita per tempat (Google):   0%|          | 0/100 [00:00<?, ?it/s]

CAPTCHA terdeteksi -> selesaikan manual di jendela Chrome...


Berita per tempat (Google):  29%|ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â°       | 29/100 [03:17<07:10,  6.06s/it]

CAPTCHA terdeteksi -> selesaikan manual di jendela Chrome...


...

Berita per tempat (Google):  92%|ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€šÃ‚Â| 92/100 [14:48<00:56,  7.04s/it]

CAPTCHA terdeteksi -> selesaikan manual di jendela Chrome...


Berita per tempat (Google): 100%|ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“Ãƒâ€¹Ã¢â‚¬Â | 100/100 [16:42<00:00, 10.02s/it]


Tempat belum punya data: 1912


,nama_tempat,keyword_berita,jumlah_berita_6bln,tanggal_berita_pertama,tanggal_berita_terakhir,selisih_hari
0,Sai Ramen,Sai Ramen Jakarta Selatan,5.0,2026-06-28,2026-10-02,96.0
1,Menya Beast,Menya Beast Jakarta Selatan,4.0,2026-09-11,2026-10-02,21.0
2,Raku Ramen,Raku Ramen Jakarta Selatan,3.0,2026-10-01,2026-10-02,1.0
3,Echigoya,Echigoya Jakarta,3.0,2026-04-22,2026-10-02,163.0
4,Warung Amanda,Warung Amanda Sidoarjo,3.0,2026-05-11,2026-09-25,137.0
...,...,...,...,...,...,...
134,Roti Bakar Eddy,Roti Bakar Eddy Jakarta Selatan,10.0,2026-04-10,2026-09-29,172.0
135,Mie Kangkung Kite,Mie Kangkung Kite Jakarta Selatan,2.0,2026-05-30,2026-09-02,95.0
136,Ayam Bakar Ganthari,Ayam Bakar Ganthari Jakarta Selatan,2.0,2026-05-14,2026-05-30,16.0
137,Ayam Bulungan,Ayam Bulungan Jakarta Selatan,19.0,2026-05-03,2026-09-25,145.0


In [27]:
import os, getpass

In [28]:
# Input API key langsung di notebook (hindari masalah env var PowerShell yang suka kemasukan
# whitespace/newline tak sengaja). Key tidak ditampilkan & tidak tersimpan ke file notebook.
import getpass

_key = getpass.getpass("Masukkan Serper API key (dari https://serper.dev/api-key): ").strip()
if _key:
    os.environ["SERPER_API_KEY"] = _key
del _key

In [37]:
# Versi Serper.dev (https://serper.dev/) -- menggantikan Selenium supaya tidak kena CAPTCHA.
# Serper adalah proxy resmi ke Google Search/News; permintaan dikirim via REST API mereka,
# jadi tidak ada browser yang perlu dibuka dan tidak ada CAPTCHA yang perlu diselesaikan manual.
#
# Setup:
#   1. Daftar gratis di https://serper.dev/ (dapat 2.500 query gratis, tanpa kartu kredit).
#   2. Set API key sebagai environment variable SERPER_API_KEY (jangan hardcode di notebook).
#      PowerShell: $env:SERPER_API_KEY = "xxxxx"
#   3. pip install requests (sudah ada di cell import di atas)
#
# Checkpoint terpisah dari versi Selenium supaya tidak tercampur / bisa dibandingkan.

SERPER_API_KEY = os.environ.get("SERPER_API_KEY", "").strip()
if not SERPER_API_KEY:
    raise RuntimeError(
        "SERPER_API_KEY belum di-set. Set environment variable SERPER_API_KEY lalu restart kernel."
    )
if len(SERPER_API_KEY) > 64 or any(ch.isspace() for ch in SERPER_API_KEY):
    raise RuntimeError(
        f"SERPER_API_KEY terlihat tidak valid (panjang={len(SERPER_API_KEY)}, ada whitespace="
        f"{any(ch.isspace() for ch in SERPER_API_KEY)}). Key asli dari dashboard serper.dev "
        "biasanya ~40 karakter hex tanpa spasi/baris baru. Cek ulang cara kamu set env var-nya "
        "(mis. jangan pakai Get-Content tanpa -Raw, jangan ada kutip/newline ikut ter-copy)."
    )
print(f"SERPER_API_KEY terbaca, panjang: {len(SERPER_API_KEY)}")

SERPER_NEWS_URL = "https://google.serper.dev/news"
PLACE_CKPT_SERPER = Path("place_news_stats_google_checkpoint.jsonl")
SERPER_PAGES = 3          # halaman hasil per tempat; berhenti lebih awal kalau sudah mentok (lihat fetch_place_news_stats_serper)
SERPER_NUM = 30           # jumlah hasil yang diminta ke Serper (endpoint /news tampaknya selalu balas ~10/halaman apa pun nilainya)
SERPER_DELAY = (0.2, 0.6)  # jeda acak antar request (detik); API-nya sendiri sudah tahan rate tinggi
SERPER_MAX_RETRIES = 3
LIMIT = 3000

_serper_session = requests.Session()
_serper_session.headers.update({
    "X-API-KEY": SERPER_API_KEY,
    "Content-Type": "application/json",
})


def build_serper_payload(query: str, page: int = 1) -> dict:
    cd_min = f"{range_start.month}/{range_start.day}/{range_start.year}"
    cd_max = f"{today.month}/{today.day}/{today.year}"
    return {
        "q": query,
        "gl": "id",
        "hl": "id",
        "tbs": f"cdr:1,cd_min:{cd_min},cd_max:{cd_max}",
        "num": SERPER_NUM,
        "page": page,
    }


def parse_serper_date(item: dict):
    """Serper mengembalikan teks tanggal yang sama seperti tampilan Google ('3 hari yang lalu',
    '5 Mei 2026', dst), jadi dipakai ulang parser yang sama. Fallback ke dateutil kalau gagal."""
    text = item.get("date", "")
    if not text:
        return None
    d = parse_news_date(text)
    if d:
        return d
    try:
        d = dateparser.parse(text, default=today)
        if d and d <= today:
            return d
    except (ValueError, OverflowError):
        pass
    return None


def fetch_place_news_stats_serper(place: str, city: str | None = None) -> dict:
    q = f'{place}'
    if USE_CITY_IN_QUERY and city:
        q += f' {city}'

    all_found = {}
    for page in range(1, SERPER_PAGES + 1):
        payload = build_serper_payload(q, page=page)
        resp = None
        last_err = None
        for attempt in range(SERPER_MAX_RETRIES):
            try:
                resp = _serper_session.post(SERPER_NEWS_URL, json=payload, timeout=20)
                if resp.status_code == 200:
                    break
                if resp.status_code == 429:
                    time.sleep(5 * (attempt + 1))
                    continue
                resp.raise_for_status()
            except requests.RequestException as e:
                last_err = e
                resp = None
                time.sleep(2 * (attempt + 1))
        if resp is None or resp.status_code != 200:
            status = resp.status_code if resp else None
            raise RuntimeError(f"Serper gagal (status={status}): {last_err}")

        items = resp.json().get("news", [])
        if not items:  # halaman kosong -> sudah mentok
            break
        new = {}
        for item in items:
            link = item.get("link")
            if not link or link in all_found:
                continue
            d = parse_serper_date(item)
            if d:
                new[link] = d
        all_found.update(new)
        if not new:  # halaman berikutnya cuma mengulang hasil yang sama -> sudah mentok
            break
        time.sleep(random.uniform(*SERPER_DELAY))

    dates = [d for d in all_found.values() if d >= range_start]
    return {
        "nama_tempat": place,
        "error": False,
        "keyword_berita": q,
        "jumlah_berita_6bln": len(dates),
        "tanggal_berita_pertama": min(dates).date().isoformat() if dates else None,
        "tanggal_berita_terakhir": max(dates).date().isoformat() if dates else None,
    }


# --- resume dari checkpoint ---
place_done_serper = {}
if PLACE_CKPT_SERPER.exists():
    with PLACE_CKPT_SERPER.open(encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not rec.get("error"):
                place_done_serper[rec["nama_tempat"]] = rec

df_todo_serper = (
    df_places_final.dropna(subset=["nama_tempat"])
    .drop_duplicates(subset="nama_tempat")[["nama_tempat", "kota_kabupaten"]]
)
todo_serper = [
    (p, clean_city(c)) for p, c in df_todo_serper.itertuples(index=False) if p not in place_done_serper
]
print(f"Sudah ada: {len(place_done_serper)} | Perlu diproses: {len(todo_serper)}")
if LIMIT:
    todo_serper = todo_serper[:LIMIT]

if todo_serper:
    with PLACE_CKPT_SERPER.open("a", encoding="utf-8") as ckpt:
        for place, city in tqdm(todo_serper, desc="Berita per tempat (Serper)"):
            try:
                rec = fetch_place_news_stats_serper(place, city)
            except Exception as e:
                print(f"Gagal '{place}': {e}")
                rec = {"nama_tempat": place, "error": True}
            ckpt.write(json.dumps(rec, ensure_ascii=False) + "\n")
            ckpt.flush()
            if not rec.get("error"):
                place_done_serper[place] = rec
            time.sleep(random.uniform(*SERPER_DELAY))

# --- gabungkan ke df_places_final (menimpa hasil versi Selenium/RSS) ---
stat_cols = ["jumlah_berita_6bln", "tanggal_berita_pertama", "tanggal_berita_terakhir"]
df_stats_serper = pd.DataFrame(place_done_serper.values())[["nama_tempat", "keyword_berita"] + stat_cols]
df_places_final = df_places_final.drop(
    columns=[c for c in df_stats_serper.columns if c != "nama_tempat"] + ["selisih_hari"], errors="ignore"
).merge(df_stats_serper, on="nama_tempat", how="left")

tgl_pertama = pd.to_datetime(df_places_final["tanggal_berita_pertama"])
tgl_terakhir = pd.to_datetime(df_places_final["tanggal_berita_terakhir"])
df_places_final["selisih_hari"] = (tgl_terakhir - tgl_pertama).dt.days

print(f"Tempat belum punya data: {df_places_final['jumlah_berita_6bln'].isna().sum()}")
df_places_final[["nama_tempat", "keyword_berita"] + stat_cols + ["selisih_hari"]].dropna(
    subset=["jumlah_berita_6bln"]
)

SERPER_API_KEY terbaca, panjang: 40
Sudah ada: 316 | Perlu diproses: 1735


Berita per tempat (Serper):   0%|          | 0/1735 [00:00<?, ?it/s]

Berita per tempat (Serper):  15%|█▍        | 260/1735 [13:18<1:16:08,  3.10s/it]

Gagal 'Pecel "Senggol" Beringharjo': Serper gagal (status=None): 400 Client Error: Bad Request for url: https://google.serper.dev/news


Berita per tempat (Serper):  63%|██████▎   | 1098/1735 [59:30<42:36,  4.01s/it]  

...

Berita per tempat (Serper): 100%|█████████▉| 1734/1735 [3:11:55<00:12, 12.49s/it]

Gagal 'Agrowisata Salak Sibetan': Serper gagal (status=None): 400 Client Error: Bad Request for url: https://google.serper.dev/news


Berita per tempat (Serper): 100%|██████████| 1735/1735 [3:12:07<00:00,  6.64s/it]

Tempat belum punya data: 638


,nama_tempat,keyword_berita,jumlah_berita_6bln,tanggal_berita_pertama,tanggal_berita_terakhir,selisih_hari
0,Sai Ramen,Sai Ramen Jakarta Selatan,5.0,2026-06-28,2026-10-02,96.0
1,Menya Beast,Menya Beast Jakarta Selatan,4.0,2026-09-11,2026-10-02,21.0
2,Raku Ramen,Raku Ramen Jakarta Selatan,3.0,2026-10-01,2026-10-02,1.0
3,Echigoya,Echigoya Jakarta,3.0,2026-04-22,2026-10-02,163.0
4,Warung Amanda,Warung Amanda Sidoarjo,3.0,2026-05-11,2026-09-25,137.0
...,...,...,...,...,...,...
1409,Kopi Kimteng,Kopi Kimteng Pekanbaru,5.0,2026-05-08,2026-07-11,64.0
1410,Pusat Pelatihan Gajah,Pusat Pelatihan Gajah,30.0,2026-04-23,2026-10-02,162.0
1411,Camping Ground Pulau Tongah,Camping Ground Pulau Tongah Kampar,2.0,2026-05-23,2026-05-23,0.0
1412,Air Terjun Batu Dinding,Air Terjun Batu Dinding Kampar,11.0,2026-05-19,2026-09-02,106.0


In [34]:
df_places_final[df_places_final["jumlah_berita_6bln"] > 10]

,no,nama_tempat,kategori,jenis_tempat,lokasi,keterangan,status_viral,viral_sejak,dasar_viral,kelurahan_desa,...,catatan_lokasi,jumlah_artikel,keyword_pencarian,judul_artikel_sumber,link_sumber,keyword_berita,jumlah_berita_6bln,tanggal_berita_pertama,tanggal_berita_terakhir,selisih_hari
7,8,TikTok Food Fest 2026,Pameran,festival kuliner,"Hutan Kota Gelora Bung Karno (GBK), Jakarta","Festival kuliner gratis 26-27 Sep 2026, 40+ tenant F&B, demo masak, musik; tema 'Every Ingredient Tells a Story'",Disebut viral,26-27 September 2026,Menghadirkan sejumlah kuliner yang populer/banyak diperbincangkan di TikTok,Gelora,...,Hutan Kota GBK,1,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral - Aktualita.co",https://news.google.com/rss/articles/CBMidEFVX3lxTE5xMXhUUkZOTGJJcWZ2VmgwdGgtcnJsZ3NnbklPWENaLWV0RTVpM2Njc25KNDhpOFZ...,TikTok Food Fest 2026 Jakarta Pusat,30.0,2026-04-16,2026-10-02,169.0
14,15,Aldi's Burger,Restoran,burger (tenant festival),"Tenant TikTok Food Fest, Hutan Kota GBK",Tenant festival (burger),Disebut viral,Tidak disebut,Tenant kuliner yang banyak diperbincangkan di TikTok,Gelora,...,"Lokasi tenant festival, bukan gerai tetap",2,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral - Aktualita.co",https://news.google.com/rss/articles/CBMidEFVX3lxTE5xMXhUUkZOTGJJcWZ2VmgwdGgtcnJsZ3NnbklPWENaLWV0RTVpM2Njc25KNDhpOFZ...,Aldi's Burger Jakarta Pusat,17.0,2026-04-06,2026-07-28,113.0
19,20,DaGo,Restoran,restoran ayam goreng Korea,"Panglima Polim, Jakarta Selatan","Ayam goreng Korea tradisional, yangnyeom, jjajangmyeon, tteokbokki; Rp89-159 ribu",Disebut viral,Tidak disebut,"""Restoran ini juga viral di media sosial lho!""",None,...,Area Jl. Panglima Polim,1,tempat makan viral,"Bukan Fried Chicken Biasa, Ini 5 Restoran Ayam Goreng Korea Enak di Jakarta - food.detik.com",https://news.google.com/rss/articles/CBMivwFBVV95cUxNWWlVUWNKY0FTdC1PTDNfLUdFV2xRVXNWc3dSZldmVnJ1QWl2VW53V3ZuTVFvYjd...,DaGo Jakarta Selatan,23.0,2026-04-10,2026-09-11,154.0
76,77,Cita Rasa Nusantara,Restoran,restoran Nusantara rumah Joglo,"Ampera Raya, Jakarta Selatan","Interior Joglo, suasana asri berumput; soto Betawi Rp79 ribu",Rekomendasi,Tidak disebut,Suasana asri dengan interior rumah Joglo,Cilandak Timur,...,Koordinat perkiraan area Ampera Raya,1,tempat makan viral,"5 Tempat Makan Enak di Ampera Raya Jaksel, Ada Nasi Bali hingga Live Seafood - food.detik.com",https://news.google.com/rss/articles/CBMiwAFBVV95cUxPLTVYWUJBdVc3a3Jjd0ZKbTFhSVJyTkJVX3B1OWtUMEtMWjRNNTJVejhUSFFkTEh...,Cita Rasa Nusantara Jakarta Selatan,30.0,2026-04-08,2026-10-01,176.0
79,80,Pasar Cihapit,lainnya,pasar tradisional kuliner,"Jl. Cihapit Jl. Sabang No.32, Cihapit, Kec. Bandung Wetan","Pasar tradisional hits dengan kedai Toko Bakmi Feng, Rama Ramen, Baso Gepeng 99",Disebut viral,Tidak disebut,"Pasar tradisional yang lagi hits, favorit wisatawan",Cihapit,...,None,1,tempat makan viral,13 Tempat Makan Hits dan Viral di Bandung Yang Wajib Kamu Kunjungi - TripZilla Indonesia,https://news.google.com/rss/articles/CBMiZ0FVX3lxTFBOYktCUGo5UEpQNWNnbkowTm0wUW1tOEU4YlBYU1gwdlRfUzBYQ0hBOTRHU19PZGJ...,Pasar Cihapit Bandung,30.0,2026-04-09,2026-09-25,169.0
81,82,Warung Nasi Ibu Imas,Restoran,warung nasi,Bandung,Buka pagi hingga tengah malam,Disebut viral,Tidak disebut,Masuk daftar tempat makan hits dan viral di Bandung,Balonggede,...,Alamat terpotong; koordinat pusat Kota Bandung,4,tempat makan viral,13 Tempat Makan Hits dan Viral di Bandung Yang Wajib Kamu Kunjungi - TripZilla Indonesia,https://news.google.com/rss/articles/CBMiZ0FVX3lxTFBOYktCUGo5UEpQNWNnbkowTm0wUW1tOEU4YlBYU1gwdlRfUzBYQ0hBOTRHU19PZGJ...,Warung Nasi Ibu Imas Bandung,21.0,2026-04-11,2026-09-26,168.0
104,105,Mie Ayam Bakso Yunus,Restoran,warung mi ayam bakso,"Tebet Barat, Jakarta Selatan","Sejak 1980-an; mie yamin, mie ayam, bakso urat besar; mulai Rp25.000",Disebut viral,Tidak disebut,Langganan artis seperti Yuni Shara dan Krisday

In [35]:
# save file
df_places_final.to_csv("df_places_final_with_news_stats.csv", index=False)

In [1]:
print("a")

a
